Code for the ADX Indicator

Notes for main integration:

Assumption is that initial ticker screening just pulls screening related information, no need to download.
Output of initial ticker screening is just a list of tickers.

For main(), loop through ticker in tickers.

In each loop, download price data for that ticker. 



In [1]:
import pandas as pd
import yfinance as yf
import numpy as np

In [2]:
ticker = "NVDA"

start_date = "2025-01-01" #start date would be 3 months? can be determined late, before the start of the actual data
end_date = "2026-01-01" 

data = yf.download(ticker, start=start_date, end=end_date)

data = data[["High", "Low", "Close"]]


[*********************100%***********************]  1 of 1 completed


In [3]:


def true_range(data):
    high_low = data["High"] - data["Low"]
    high_close = abs(data["High"] - data["Close"].shift(1))
    low_close = abs(data["Low"] - data["Close"].shift(1))
    
    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close
    
    data["True Range"] = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    
    return data


true_range(data)
print(data)

Price             High         Low       Close Trading Range    up gap  \
Ticker            NVDA        NVDA        NVDA                           
Date                                                                     
2025-01-02  138.517877  134.278959  137.949356      4.238918       NaN   
2025-01-03  144.522194  139.365676  144.093323      5.156518  6.572838   
2025-01-06  151.763283  147.434602  149.040390      4.328681  7.669960   
2025-01-07  152.730732  139.644931  139.774597     13.085801  3.690342   
2025-01-08  143.574659  137.201321  139.744675      6.373338  3.800061   
...                ...         ...         ...           ...       ...   
2025-12-24  188.468980  186.154389  188.169678      2.314591  0.299315   
2025-12-26  192.240151  187.561098  190.085190      4.679053  4.070473   
2025-12-29  188.319327  185.475989  187.780594      2.843337  1.765863   
2025-12-30  188.548800  186.493596  187.102173      2.055204  0.768206   
2025-12-31  190.115141  186.054650  18

In [4]:
def directional_momentum(data):
    up_move = data["High"].diff() 
    down_move = -data["Low"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)
    
    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    return data

directional_momentum(data)
print(data)
    

Price             High         Low       Close Trading Range    up gap  \
Ticker            NVDA        NVDA        NVDA                           
Date                                                                     
2025-01-02  138.517877  134.278959  137.949356      4.238918       NaN   
2025-01-03  144.522194  139.365676  144.093323      5.156518  6.572838   
2025-01-06  151.763283  147.434602  149.040390      4.328681  7.669960   
2025-01-07  152.730732  139.644931  139.774597     13.085801  3.690342   
2025-01-08  143.574659  137.201321  139.744675      6.373338  3.800061   
...                ...         ...         ...           ...       ...   
2025-12-24  188.468980  186.154389  188.169678      2.314591  0.299315   
2025-12-26  192.240151  187.561098  190.085190      4.679053  4.070473   
2025-12-29  188.319327  185.475989  187.780594      2.843337  1.765863   
2025-12-30  188.548800  186.493596  187.102173      2.055204  0.768206   
2025-12-31  190.115141  186.054650  18

In [8]:
def wilder_smoothing(data, period):

    smoothed = {}
    initial = sum(data.values[1:period+1])
    smoothed[period] = initial

    for i in range(period+1, len(data)):
        smoothed[i] = smoothed[i-1] - (smoothed[i-1]/period) + data.values[i]

    return pd.Series(list(smoothed.values()), index=data.index[period:])
    

In [9]:
period = 14 #adjustable

data["smoothed +DM"]= wilder_smoothing(data["+DM"], period)
data["smoothed -DM"]=wilder_smoothing(data["-DM"], period)
data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

In [10]:
def directional_index(data):
    data["+DI"] = (data["smoothed +DM"]/data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"]/data["smoothed TR"]) * 100

    data["DX"] = 100* (abs(data["+DI"] - data["-DI"]))/((abs(data["+DI"])) + abs(data["-DI"]))


    
    return data

def adx(data, period):
    adx = {}
    initial = (data["DX"].iloc[period:2*period]).mean()

    adx[2*period-1] = initial

    for i in range(2*period, len(data)):
        adx[i] = ((period-1) * (adx[i-1]) + data["DX"].iloc[i])/period


    data["adx"] = pd.Series(list(adx.values()), index=data.index[2*period-1:])
    return data



directional_index(data)
adx(data, period=14)
print(data[["+DI", "-DI", "DX", "adx"]].tail(10))
    

Price             +DI        -DI         DX        adx
Ticker                                                
Date                                                  
2025-12-17  17.516155  29.711897  25.823089  12.885059
2025-12-18  16.478920  27.913093  25.757274  13.804503
2025-12-19  21.174425  25.573654   9.410502  13.490645
2025-12-22  23.596928  24.610387   2.102291  12.677191
2025-12-23  27.915702  22.739394  10.218730  12.501587
2025-12-24  27.014487  22.005288  10.218730  12.338526
2025-12-26  30.153796  20.716032  18.552773  12.782401
2025-12-29  28.389306  22.150966  12.343305  12.751037
2025-12-30  27.918561  21.545557  12.884095  12.760541
2025-12-31  28.504207  20.361496  16.663448  13.039320
